In [13]:
import warnings
import yfinance as yf
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, timedelta
import numpy as np

# Configuração para ignorar avisos desnecessários de visualização
warnings.filterwarnings("ignore")

# ==========================================
# CONFIGURAÇÕES E PARÂMETROS DO PROJETO
# ==========================================

acoes = ['ITUB4.SA', 'PETR4.SA', 'VALE3.SA', 'IVVB11.SA']
indices = ['^BVSP', '^GSPC', 'BRL=X', 'GC=F']

lista_tickers = acoes + indices

dict_carteira = {
    'ITUB4.SA': 5000,
    'PETR4.SA': 3000,
    'VALE3.SA': 4000,
    'IVVB11.SA': 6000
}

# Definição do período de análise (últimos 2 anos)
data_inicio = (datetime.now() - timedelta(days=730)).strftime('%Y-%m-%d')
data_fim = datetime.now().strftime('%Y-%m-%d')

In [14]:
# ==========================================
# FUNÇÕES DE EXTRAÇÃO E PROCESSAMENTO
# ==========================================


def pegar_cotacoes(tickers:list[str], start:    str, end:str) -> pd.DataFrame:
    """Obtém dados de fechamento ajustado do Yahoo Finance, trata nulos e retornos vazios."""
    df = yf.download(tickers, start, end , auto_adjust=False)

    if 'Adj Close' not in df.columns:
        raise ValueError('Aviso: Coluna "Adj Close" não encontrada.')
    
    df_limpo = df['Adj Close'].ffill().dropna()

    if df_limpo.empty:
        raise ValueError('Nenhum dado válido foi retornado.')
    
    return df_limpo


def criar_carteira(carteira: dict[str,float], cotacoes: pd.DataFrame) -> pd.DataFrame:
    """Gera o histórico de patrimônio da carteira de forma vetorizada e limpa."""
    # Garante que estamos operando apenas com as colunas necessárias
    ativos_validos = [col for col in carteira if col in cotacoes.columns]
    preco_inicial_ativo = cotacoes[ativos_validos].iloc[0]

    # Vetorização: Calcula a quantidade de cotas de uma só vez
    shares = pd.Series(carteira) / preco_inicial_ativo

    # Calcula a evolução patrimonial de cada ativo e o Total
    df_carteira = cotacoes[ativos_validos].multiply(shares, axis=1)
    df_carteira['Total'] = df_carteira.sum(axis=1)

    return df_carteira


def conversao_cambial(df: pd.DataFrame) -> pd.DataFrame:
    """Aplica as conversões cambiais para ativos dolarizados usando .assign para evitar fatiamento."""
    return df.assign(
        **{
            "SP500 (R$)": df["^GSPC"] * df["BRL=X"],
            "Ouro (R$)": df["GC=F"] * df["BRL=X"],
            "Dólar": df["BRL=X"],
        }
    ).drop(columns=["^GSPC", "GC=F", "BRL=X"])


def calcular_retorno(df: pd.DataFrame) -> pd.Series:
    """Calcula o retorno percentual acumulado do período simplificado."""
    return ((df.iloc[-1] / df.iloc[0]) - 1) * 100



In [15]:
# ==========================================
# EXECUÇÃO DO PIPELINE DE DADOS
# ==========================================

# 1. Carga de dados
df_cotacoes_cru = pegar_cotacoes(
    tickers=lista_tickers, 
    start=data_inicio, 
    end=data_fim
    )

# 2. Processamento da carteira e moedas (Usando cópias explícitas)
df_cotacoes = conversao_cambial(
    df=df_cotacoes_cru.copy()
    )

df_carteira = criar_carteira(
    carteira=dict_carteira, 
    cotacoes=df_cotacoes.copy()
    )

# 3. Exibição dos Retornos Calculados
print("\n--- Retorno Acumulado da Carteira (%) ---")
print(calcular_retorno(df_carteira).round(2))

print("\n--- Retorno Acumulado dos Ativos e Benchmarks (%) ---")
print(calcular_retorno(df_cotacoes).round(2))

[*********************100%***********************]  8 of 8 completed


--- Retorno Acumulado da Carteira (%) ---
Ticker
ITUB4.SA     59.78
PETR4.SA     59.74
VALE3.SA     49.34
IVVB11.SA    40.32
Total        50.96
dtype: float64

--- Retorno Acumulado dos Ativos e Benchmarks (%) ---
Ticker
ITUB4.SA      59.78
IVVB11.SA     40.32
PETR4.SA      59.74
VALE3.SA      49.34
^BVSP         38.34
SP500 (R$)    36.31
Ouro (R$)     84.53
Dólar         -2.43
dtype: float64


In [16]:
# ==========================================
# COMPARAÇÃO E NORMALIZAÇÃO DE RENTABILIDADE
# ==========================================

def gerar_comparacao(carteira:pd.DataFrame, cotacoes:pd.DataFrame) -> pd.DataFrame:
    """Consolida a carteira com os benchmarks principais e normaliza os dados
    na Base 100 para uma comparação visual e estatística justa.
    """
    # Isola o retorno patrimonial total da carteira
    df_comparacao = pd.DataFrame(
        {'Carteira total': carteira['Total']}, index=carteira.index
    )
    
    # Adiciona os índices que deseja comparar (ex: Ibovespa e S&P500 em R$)
    benchmarks = ["^BVSP", "SP500 (R$)"]
    for bench in benchmarks:
        if bench in cotacoes.columns:
            df_comparacao[bench] = cotacoes[bench]

    # VETORIZAÇÃO: Transforma todas as séries temporais para a Base 100 (Dia Inicial = 100)
    # Isso permite comparar uma carteira de R$18k com o Ibovespa a 120k pontos.
    df_normalizado = (df_comparacao / df_comparacao.iloc[0]) * 100

    return df_normalizado

df_comparacao = gerar_comparacao(
    carteira=df_carteira, 
    cotacoes=df_cotacoes
    )

# ==========================================
# VISUALIZAÇÃO GRÁFICA PROFISSIONAL (PLOTLY)
# ==========================================


fig = px.line(
    df_comparacao, 
    x=df_comparacao.index, 
    y=df_comparacao.columns, 
    title='Desempenho Relativo: Carteira vs. Benchmarks de Mercado (Base 100)')

fig.update_layout(
    template= 'plotly_dark',
    xaxis_title= 'Data de Análise',
    yaxis_title= 'Evolução percentual (Base 100)',
    yaxis_ticksuffix="%",
    title_font_size=18,
    hovermode='x unified',
    legend_title_text="Ativos / Índices"
    )

fig.show()

In [17]:
# ==========================================
# ANÁLISE DE RISCO: CORRELAÇÃO E VOLATILIDADE
# ==========================================


df_cotacoes_correlacao = df_cotacoes.copy()
df_cotacoes_correlacao['Carteira'] = df_carteira['Total']

# Cálculo de retornos logarítmicos diários de forma vetorizada
tabela_rentabilidade_diaria = np.log(
    df_cotacoes_correlacao / df_cotacoes_correlacao.shift(1)
    ).dropna()

#Matrix de correlação
tabela_correlacao = tabela_rentabilidade_diaria.corr()

fig_correlacao = px.imshow(
    tabela_correlacao, 
    text_auto='.2f', 
    color_continuous_scale='RdBu_r',
    zmin=-1,
    zmax=1,
    title='Matrix de Correlação dos Retornos Diários')

fig_correlacao.update_layout(
    template= 'plotly_dark',
    xaxis_title= 'Ativos / Índices',
    yaxis_title= 'Ativos / Índices',
    title_font_size=20
)

fig_correlacao.show()


#Análise de volatilidade

tabela_volatividade = (
    tabela_rentabilidade_diaria.std() * np.sqrt(252) * 100).to_frame(name='Volatilidade Anualizada (%)').round(2)
display(tabela_volatividade)



,Volatilidade Anualizada (%)
Ticker,
ITUB4.SA,21.06
IVVB11.SA,15.65
PETR4.SA,23.77
VALE3.SA,24.31
^BVSP,15.53
SP500 (R$),21.55
Ouro (R$),25.87
Dólar,13.16
Carteira,11.65


### Análise Técnica e Indicadores

In [18]:
data_inicio_analise = (datetime.now() - timedelta(days=365)).strftime('%Y-%m-%d')
data_fim_analise = datetime.now().strftime('%Y-%m-%d')


ticker = 'VALE3.SA' # Altere aqui caso queira análise de outra cotação

df_analise = yf.download(
    ticker, 
    data_inicio_analise, 
    data_fim_analise, 
    multi_level_index=False
    )

df_analise['MM50'] = df_analise['Close'].rolling(50).mean() # Média movel de 50 dias
df_analise['MM200'] = df_analise['Close'].rolling(200).mean() # Média movel 200 dias

# Plotagem de resultados
fig_analise = go.Figure()

fig_analise.add_trace(
    go.Candlestick(
        x=df_analise.index, 
        open=df_analise['Open'], 
        close=df_analise['Close'], 
        high=df_analise['High'], 
        low=df_analise['Low'],
        name='Preço de Mercado'
        ))

fig_analise.add_trace(
    go.Scatter(
        x=df_analise.index, 
        y=df_analise['MM50'], 
        name='Média movel 50 dias',
        line={'color':'blue', 'width': 1.5}
        ))

fig_analise.add_trace(
    go.Scatter(
        x=df_analise.index, 
        y=df_analise['MM200'], 
        name='Média movel 200 dias',
        line={'color':'orange', 'width': 1.5}
        ))

fig_analise.update_layout(
    template='plotly_dark',
    title=f'Análise Técnica: Histórico e Médias Móveis de {ticker}',
    xaxis_title='Data',
    yaxis_title='Preços (R$)',
    title_font_size=20,
    xaxis_rangeslider_visible=False,
    hovermode='x unified',
    )

fig_analise.show()

[*********************100%***********************]  1 of 1 completed
